# Class 2: Preprocessing, word probabilities, and log odds

How do decisions about text representation affect the things we measure?

This notebook follows the class 2 slides. Work through the small examples alongside the lecture; Part 6 is a longer practice task using the four speeches from class 1.

1. Preprocessing as measurement
2. Word features and document frequency
3. The multinomial model of language
4. Odds and log odds
5. TF-IDF and similarity
6. Sensitivity on real speeches

**Setup:** Python 3.9 or later, using only the standard library. No packages, model downloads, API keys, or internet access are needed for the calculations. Run cells in order. Saved outputs show the supplied examples; after changing a setting, rerun the affected section.

All short sentences are invented classroom examples. The John/Mary counts come from the earlier course materials. The speech sample is stored in `data/sotu_sample.csv`.


## Setup

These helpers print tables and compare sparse count/weight vectors. A vector is stored as a mapping from a feature to its value: missing features have value zero. Cosine is undefined for an empty or all-zero vector; we return `None` and display it as `undefined`.


In [1]:
from collections import Counter
from itertools import combinations
from pathlib import Path
from math import log, exp, sqrt, lgamma, comb, isclose
import csv
import re

def show_table(headers, rows):
    rows = [[str(value) for value in row] for row in rows]
    widths = [max([len(str(h))] + [len(row[i]) for row in rows])
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows:
        print("  ".join(value.ljust(w) for value, w in zip(row, widths)))

def cosine(x, y):
    nx = sqrt(sum(value ** 2 for value in x.values()))
    ny = sqrt(sum(value ** 2 for value in y.values()))
    if nx == 0 or ny == 0:
        return None
    return sum(value * y.get(term, 0) for term, value in x.items()) / (nx * ny)

def number(value):
    return "undefined" if value is None else f"{value:.3f}"


## 1. Preprocessing as measurement

A preprocessing rule decides which distinctions remain available to the analysis. Removing pronouns might be reasonable for some questions about policy content, but problematic for studying group identity.

### Tokenization, case, punctuation, and numbers

Inspect the alternatives below. Our word tokenizer extracts English letter sequences and retains internal apostrophes and hyphens. It is a teaching rule, not a general multilingual tokenizer. Notice that retaining a contraction as a token does not resolve its meaning.


In [2]:
example = "We can't cut 2,000 public-sector jobs!"
WORD_PATTERN = r"[A-Za-z]+(?:['-][A-Za-z]+)*"

def word_tokens(text, lowercase=True):
    found = re.findall(WORD_PATTERN, text)
    return [w.lower() for w in found] if lowercase else found

print("Whitespace:", example.split())
print("Word tokens, preserve case:", word_tokens(example, lowercase=False))
print("Word tokens, lowercase:", word_tokens(example))
print("Words, numbers, and punctuation:",
      re.findall(r"[A-Za-z]+(?:['-][A-Za-z]+)*|[0-9]+(?:,[0-9]{3})*|[^\w\s]", example))


Whitespace: ['We', "can't", 'cut', '2,000', 'public-sector', 'jobs!']
Word tokens, preserve case: ['We', "can't", 'cut', 'public-sector', 'jobs']
Word tokens, lowercase: ['we', "can't", 'cut', 'public-sector', 'jobs']
Words, numbers, and punctuation: ['We', "can't", 'cut', '2,000', 'public-sector', 'jobs', '!']


**Pause and explain.** Which representation would you use to study (a) attention to employment, (b) the number of threatened positions, or (c) rhetorical emphasis? What would each discard?

Your answer:

### Stemming and lemmatization

Stemming applies shortening rules; lemmatization identifies a dictionary form, potentially using part of speech and context. Neither automatically groups all synonyms.

The following is an **explicit, limited replacement dictionary**, not a stemmer or a lemmatizer. It demonstrates the effect of combining selected forms without installing a language model.


In [3]:
forms = ["job", "jobs", "worker", "workers", "work", "worked", "working", "employment"]
replacement = {"jobs": "job", "workers": "worker", "worked": "work", "working": "work"}
combined = [replacement.get(word, word) for word in forms]
show_table(["original", "combined"], zip(forms, combined))
print("Types before:", len(set(forms)), "| Types after:", len(set(combined)))


original    combined  
----------  ----------
job         job       
jobs        job       
worker      worker    
workers     worker    
work        work      
worked      work      
working     work      
employment  employment
Types before: 8 | Types after: 4


**Explain.** Why does `employment` remain separate? When might combining `worked` and `working` remove information you care about?

Your answer:

### Stopwords and negation

Predict what remains with list A and list B. The lists are deliberately small; neither is a recommended general English stopword list.


In [4]:
claims = ["We are creating jobs.", "We are not creating jobs.", "They are creating jobs!"]
stop_A = {"we", "they", "are"}
stop_B = stop_A | {"not"}

def remove_stopwords(tokens, stopwords):
    return [word for word in tokens if word not in stopwords]

for label, stopwords in [("A", stop_A), ("B", stop_B)]:
    print(f"List {label}:")
    for i, text in enumerate(claims, 1):
        print(i, remove_stopwords(word_tokens(text), stopwords))
    print()


List A:
1 ['creating', 'jobs']
2 ['not', 'creating', 'jobs']
3 ['creating', 'jobs']

List B:
1 ['creating', 'jobs']
2 ['creating', 'jobs']
3 ['creating', 'jobs']



**Explain.** Which sentences become identical? Would you use the same list for *attention to employment* and *claims that jobs are being created*?

Your answer:

### A change in the thing we measure

The next example reproduces the pronoun comparison in the slides. Predict whether documents 1 and 2 or documents 1 and 3 will be more similar after removing pronouns.


In [5]:
identity_texts = ["we support jobs", "they support jobs", "we oppose jobs"]
keep_vectors = [Counter(word_tokens(text)) for text in identity_texts]
drop_vectors = [Counter(remove_stopwords(word_tokens(text), {"we", "they"}))
                for text in identity_texts]
show_table(["pair", "keep pronouns", "remove pronouns"],
           [[f"D1-D{j + 1}", number(cosine(keep_vectors[0], keep_vectors[j])),
             number(cosine(drop_vectors[0], drop_vectors[j]))] for j in [1, 2]])


pair   keep pronouns  remove pronouns
-----  -------------  ---------------
D1-D2  0.667          1.000          
D1-D3  0.667          0.500          


**Interpret.** What distinction makes documents 1 and 3 similar before removal? What distinction makes documents 1 and 2 identical afterward? Neither cosine is automatically a valid measure of stance or identity.

Your answer:


## 2. Word features and document frequency

### N-grams and the order of operations

An n-gram records consecutive tokens. After deleting words, “consecutive” means consecutive in the remaining sequence. That can create pairs that were never adjacent in the original text.


In [6]:
def ngrams(tokens, n):
    if n < 1:
        raise ValueError("n must be at least 1")
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

phrase = ["not", "in", "work"]
print("Original bigrams:", ngrams(phrase, 2))
print("Remove 'in' first:", ngrams(remove_stopwords(phrase, {"in"}), 2))
print("Trigrams:", ngrams(["we", "need", "more", "jobs"], 3))


Original bigrams: ['not in', 'in work']
Remove 'in' first: ['not work']
Trigrams: ['we need more', 'need more jobs']


**Try it.** Replace the phrase with `we are not creating jobs`. Compare removing `are` before forming bigrams with retaining the original bigrams. Describe the difference.

### Counts versus document frequency

We now use the same three-document corpus as the TF-IDF slides. Document frequency counts how many documents contain a term, regardless of repetition within a document.


In [7]:
toy_texts = ["policy jobs jobs", "policy wages", "policy wages prices"]
toy_counts = [Counter(text.split()) for text in toy_texts]
vocabulary = ["policy", "jobs", "wages", "prices"]

def document_frequency(rows):
    return Counter(term for row in rows for term, count in row.items() if count > 0)

toy_df = document_frequency(toy_counts)
show_table(["document"] + vocabulary,
           [[f"D{i + 1}"] + [row[t] for t in vocabulary]
            for i, row in enumerate(toy_counts)])
print()
show_table(["term", "total count", "document frequency"],
           [[t, sum(row[t] for row in toy_counts), toy_df[t]] for t in vocabulary])


document  policy  jobs  wages  prices
--------  ------  ----  -----  ------
D1        1       2     0      0     
D2        1       0     1      0     
D3        1       0     1      1     

term    total count  document frequency
------  -----------  ------------------
policy  3            3                 
jobs    2            1                 
wages   2            2                 
prices  1            1                 


**Predict.** Keep only terms occurring in at least two documents but not in every document. Which terms survive? Which document becomes empty?


In [8]:
MIN_DF = 2
MAX_DF = len(toy_counts) - 1
retained = [t for t in vocabulary if MIN_DF <= toy_df[t] <= MAX_DF]
filtered = [Counter({t: row[t] for t in retained if row[t] > 0}) for row in toy_counts]
print("Retained:", retained)
print("Filtered rows:", filtered)
print("Cosine D1-D2:", number(cosine(filtered[0], filtered[1])))


Retained: ['wages']
Filtered rows: [Counter(), Counter({'wages': 1}), Counter({'wages': 1})]
Cosine D1-D2: undefined


**Explain.** Is the resulting vocabulary useful for a question about jobs? An empty document is a result to investigate, not a similarity of zero.

Your answer:


## 3. The multinomial model of language

Counts describe observed words. A probability model asks how likely a count vector is, conditional on its length and word probabilities.

Assume a fixed vocabulary and independent token draws with the same probabilities. The word probabilities sum to 1. Word order does not affect the resulting count-vector probability.

### One word versus all other words: binomial

If each of three draws has probability 0.5 of producing banana, calculate the probability of exactly two bananas.


In [9]:
n, k, banana_probability = 3, 2, 0.5
binomial_probability = (comb(n, k) * banana_probability ** k
                        * (1 - banana_probability) ** (n - k))
print(f"P(exactly {k} bananas in {n} tokens) = {binomial_probability:.3f}")


P(exactly 2 bananas in 3 tokens) = 0.375


### Open the “everything else” category

The binomial example groups chocolate, fudge, and ice-cream together. Keep those words separate, with probabilities **0.5, 0.2, 0.2, 0.1** for banana, chocolate, fudge, and ice-cream respectively. These illustrative probabilities are separate from the John/Mary estimates below.

Exactly two bananas in three tokens can occur in three mutually exclusive cases:

- Two bananas and one chocolate: `(2, 1, 0, 0)`.
- Two bananas and one fudge: `(2, 0, 1, 0)`.
- Two bananas and one ice-cream: `(2, 0, 0, 1)`.

Each case has three possible token orders. Its multinomial probability is therefore `3 × P(banana)² × P(the other word)`.

**Predict:** should the three probabilities add to the binomial probability above? Why?


In [10]:
bridge_p = {"banana": 0.5, "chocolate": 0.2, "fudge": 0.2, "ice-cream": 0.1}
bridge_cases = [("chocolate", (2, 1, 0, 0)),
                ("fudge", (2, 0, 1, 0)),
                ("ice-cream", (2, 0, 0, 1))]
bridge_masses = [3 * bridge_p["banana"] ** 2 * bridge_p[word]
                 for word, counts in bridge_cases]
show_table(["other word", "count vector", "probability"],
           [[word, counts, f"{mass:.3f}"]
            for (word, counts), mass in zip(bridge_cases, bridge_masses)])
print(f"Sum of multinomial cases: {sum(bridge_masses):.3f}")
print(f"Binomial probability:     {binomial_probability:.3f}")
print("Same result:", isclose(sum(bridge_masses), binomial_probability))


other word  count vector  probability
----------  ------------  -----------
chocolate   (2, 1, 0, 0)  0.150      
fudge       (2, 0, 1, 0)  0.150      
ice-cream   (2, 0, 0, 1)  0.075      
Sum of multinomial cases: 0.375
Binomial probability:     0.375
Same result: True


The other-word probabilities sum to `0.2 + 0.2 + 0.1 = 0.5 = 1 − P(banana)`. Adding the three cases recovers the binomial calculation.

**The multinomial model uses the same independent draws but keeps track of every word category, instead of one word versus everything else.** Next, estimate those category probabilities from observed texts and use the general formula.


### Estimate John and Mary's word probabilities

Divide each speaker's pooled word counts by their total tokens. These are unsmoothed maximum-likelihood estimates for this toy model.


In [11]:
fruit_vocabulary = ["banana", "chocolate", "fudge", "ice-cream"]
john_counts = [7, 4, 1, 0]
mary_counts = [1, 3, 7, 5]

def word_probabilities(counts, alpha=0.0):
    if not counts or alpha < 0 or any(c < 0 or int(c) != c for c in counts):
        raise ValueError("Use nonnegative integer counts and nonnegative alpha")
    denominator = sum(counts) + alpha * len(counts)
    if denominator == 0:
        raise ValueError("No observations or smoothing mass")
    return [(c + alpha) / denominator for c in counts]

john_p = word_probabilities(john_counts)
mary_p = word_probabilities(mary_counts)
show_table(["term", "John count", "John p", "Mary count", "Mary p"],
           [[t, j, f"{pj:.4f}", m, f"{pm:.4f}"]
            for t, j, pj, m, pm in zip(fruit_vocabulary, john_counts, john_p, mary_counts, mary_p)])
print("Probability sums:", sum(john_p), sum(mary_p))


term       John count  John p  Mary count  Mary p
---------  ----------  ------  ----------  ------
banana     7           0.5833  1           0.0625
chocolate  4           0.3333  3           0.1875
fudge      1           0.0833  7           0.4375
ice-cream  0           0.0000  5           0.3125
Probability sums: 1.0 1.0


### The multinomial probability

For counts $x=(x_1,\ldots,x_K)$ with total $n$:

$$P(X=x\mid n,p)=\frac{n!}{\prod_v x_v!}\prod_v p_v^{x_v}.$$

The factorial coefficient counts distinct sequences with the same counts. For two bananas and two chocolates there are six orders.

We calculate in log space to avoid multiplying many tiny probabilities. `lgamma(m + 1)` equals $\ln(m!)$ for a nonnegative integer $m$. A word with positive count but model probability zero makes the entire text probability zero.


In [12]:
def multinomial_log_probability(counts, probabilities):
    if len(counts) != len(probabilities) or not counts:
        raise ValueError("Counts and probabilities must use the same vocabulary")
    if any(c < 0 or int(c) != c for c in counts):
        raise ValueError("Counts must be nonnegative integers")
    if any(not 0 <= p <= 1 for p in probabilities) or not isclose(sum(probabilities), 1.0):
        raise ValueError("Probabilities must be nonnegative and sum to 1")
    log_coefficient = lgamma(sum(counts) + 1) - sum(lgamma(c + 1) for c in counts)
    log_terms = 0.0
    for c, p in zip(counts, probabilities):
        if c == 0:
            continue  # An unobserved category contributes a factor of 1.
        if p == 0:
            return float("-inf")
        log_terms += c * log(p)
    return log_coefficient + log_terms

new_counts = [2, 2, 0, 0]
for name, probabilities in [("John", john_p), ("Mary", mary_p)]:
    log_p = multinomial_log_probability(new_counts, probabilities)
    print(f"{name}: P(counts | model) = {exp(log_p):.6f}; log probability = {log_p:.3f}")

one_order = john_p[0] ** 2 * john_p[1] ** 2
print(f"One particular ordering under John's model: {one_order:.6f}")
print(f"All six orders: {6 * one_order:.6f}")


John: P(counts | model) = 0.226852; log probability = -1.483
Mary: P(counts | model) = 0.000824; log probability = -7.101
One particular ordering under John's model: 0.037809
All six orders: 0.226852


**Explain.** Why are the two probabilities different? Is `0.226852` the probability that John wrote the text?

Your answer:

<details><summary>Check the interpretation after answering</summary>

The fitted speakers have different word probabilities. The output is the probability of these counts *under a speaker model*. It is not an authorship probability given the text. That inference needs Bayes' rule, speaker priors, and decisions about parameter uncertainty.

</details>

### Unseen words and smoothing

John's training sample has no `ice-cream`. Try the new text `ice-cream fudge fudge`, represented as `[0, 0, 2, 1]`. Then add one to every vocabulary count:

$$\widetilde p_v=\frac{c_v+\alpha}{m+\alpha K},$$

where $m$ is the number of training tokens and $K$ the vocabulary size. Add-one smoothing ($\alpha=1$) is an illustration, not a universal choice.


In [13]:
unseen_example = [0, 0, 2, 1]
ALPHA = 1.0  # Try 0, 0.1, and 1; rerun this cell.
smoothed_john_p = word_probabilities(john_counts, alpha=ALPHA)
print("Unsmoothed probability:", exp(multinomial_log_probability(unseen_example, john_p)))
print("Smoothed word probabilities:", [round(p, 4) for p in smoothed_john_p])
print("Smoothed text probability:", exp(multinomial_log_probability(unseen_example, smoothed_john_p)))
print("Ice-cream probability:", smoothed_john_p[3])


Unsmoothed probability: 0.0
Smoothed word probabilities: [0.5, 0.3125, 0.125, 0.0625]
Smoothed text probability: 0.0029296875000000035
Ice-cream probability: 0.0625


**Explain.** Does never observing a word mean it is impossible? What assumption changes as you increase `ALPHA`? What would you do with a word outside the fixed vocabulary?

Your answer:

Smoothing known categories does not automatically handle words outside the vocabulary. That requires a separate rule, such as an unknown-word category.


## 4. Odds and log odds

For a word with token probability $p$, odds compare that word with **all other tokens**:

$$\mathrm{odds}=\frac{p}{1-p},\qquad \mathrm{log\ odds}=\ln\left(\frac{p}{1-p}\right).$$

A difference in log odds is the logarithm of an odds ratio. It is not the logarithm of a probability ratio.


In [14]:
def log_odds(p):
    if not 0 < p < 1:
        return None
    return log(p / (1 - p))

show_table(["p", "odds", "log odds"],
           [[p, f"{p / (1 - p):.3f}", number(log_odds(p))] for p in [0.2, 0.5, 0.8]])

john_banana = john_p[0]
mary_banana = mary_p[0]
difference = log_odds(john_banana) - log_odds(mary_banana)
print(f"Banana log-odds difference, John minus Mary: {difference:.3f}")
print(f"Odds ratio: {exp(difference):.3f}")
print(f"Probability ratio: {john_banana / mary_banana:.3f}")


p    odds   log odds
---  -----  --------
0.2  0.250  -1.386  
0.5  1.000  0.000   
0.8  4.000  1.386   
Banana log-odds difference, John minus Mary: 3.045
Odds ratio: 21.000
Probability ratio: 9.333


**Explain.** What does the positive difference mean? Why are the odds ratio and probability ratio different? What happens to the sign if you swap John and Mary?

Your answer:

### Compare every word, with and without smoothing

At $p=0$ or $p=1$, log odds are not finite. We display `undefined` rather than silently inventing a value. Add-one smoothing over the four categories permits finite comparisons.


In [15]:
john_smooth = word_probabilities(john_counts, alpha=1)
mary_smooth = word_probabilities(mary_counts, alpha=1)
rows = []
for i, term in enumerate(fruit_vocabulary):
    a, b = log_odds(john_p[i]), log_odds(mary_p[i])
    raw = None if a is None or b is None else a - b
    adjusted = log_odds(john_smooth[i]) - log_odds(mary_smooth[i])
    rows.append([term, number(raw), number(adjusted)])
show_table(["term", "raw John-minus-Mary", "add-one John-minus-Mary"], rows)


term       raw John-minus-Mary  add-one John-minus-Mary
---------  -------------------  -----------------------
banana     3.045                2.197                  
chocolate  0.773                0.598                  
fudge      -2.147               -1.540                 
ice-cream  undefined            -1.861                 


These differences describe relative word use. They are **not** standard errors, significance tests, or uncertainty-standardized scores. Small counts can produce unstable comparisons even after smoothing.


## 5. TF-IDF and similarity

We have used counts to estimate word probabilities and compare groups. Now we use counts to construct weights for document comparison. TF-IDF is a weighting scheme, not a probability model.

For the hand calculation in the slides:

$$\mathrm{tf}_{dt}=c_{dt},\qquad
\mathrm{idf}_t=\ln\left(\frac{N}{\mathrm{df}_t}\right),\qquad
w_{dt}=c_{dt}\,\mathrm{idf}_t.$$

Use raw counts as TF, natural logarithms, and no row normalization. Only observed vocabulary terms are included, so document frequency is positive.


In [16]:
def tfidf(rows, convention="classroom"):
    if not rows:
        raise ValueError("Need at least one document")
    df = document_frequency(rows)
    n = len(rows)
    if convention == "classroom":
        idf = {term: log(n / frequency) for term, frequency in df.items()}
    elif convention == "smoothed_l2":
        idf = {term: log((1 + n) / (1 + frequency)) + 1
               for term, frequency in df.items()}
    else:
        raise ValueError("Unknown convention")
    weighted = []
    for row in rows:
        values = {term: count * idf[term] for term, count in row.items() if count > 0}
        if convention == "smoothed_l2":
            length = sqrt(sum(value ** 2 for value in values.values()))
            if length:
                values = {term: value / length for term, value in values.items()}
        weighted.append(values)
    return idf, weighted

toy_idf, toy_weights = tfidf(toy_counts)
show_table(["term", "df", "idf"],
           [[t, toy_df[t], f"{toy_idf[t]:.3f}"] for t in vocabulary])
print()
show_table(["document"] + vocabulary,
           [[f"D{i + 1}"] + [f"{row.get(t, 0):.3f}" for t in vocabulary]
            for i, row in enumerate(toy_weights)])


term    df  idf  
------  --  -----
policy  3   0.000
jobs    1   1.099
wages   2   0.405
prices  1   1.099

document  policy  jobs   wages  prices
--------  ------  -----  -----  ------
D1        0.000   2.197  0.000  0.000 
D2        0.000   0.000  0.405  0.000 
D3        0.000   0.000  0.405  1.099 


**Calculate before looking back at the output.** What is the weight of `jobs` in document 1? Why is `policy` zero even though it occurs in all three texts?

Your answer:


In [17]:
show_table(["pair", "count cosine", "TF-IDF cosine"],
           [[f"D{i + 1}-D{j + 1}", number(cosine(toy_counts[i], toy_counts[j])),
             number(cosine(toy_weights[i], toy_weights[j]))]
            for i, j in combinations(range(len(toy_counts)), 2)])


pair   count cosine  TF-IDF cosine
-----  ------------  -------------
D1-D2  0.316         0.000        
D1-D3  0.258         0.000        
D2-D3  0.816         0.346        


### The reference corpus matters

Add a document without changing the original three documents. Predict whether the original documents' IDF weights change.


In [18]:
expanded_counts = toy_counts + [Counter("jobs jobs jobs".split())]
expanded_idf, _ = tfidf(expanded_counts)
show_table(["term", "original IDF", "expanded-corpus IDF"],
           [[t, f"{toy_idf[t]:.3f}", f"{expanded_idf[t]:.3f}"] for t in vocabulary])


term    original IDF  expanded-corpus IDF
------  ------------  -------------------
policy  0.000         0.288              
jobs    1.099         0.693              
wages   0.405         0.693              
prices  1.099         1.386              


**Explain.** Did the first document change, or did the comparison corpus change? Why would this matter for comparing scores across separate analyses?

### A different software convention

The formula below reproduces scikit-learn's default TF-IDF weighting of an **already constructed count matrix**, without requiring scikit-learn. It does not reproduce its tokenizer or vocabulary filtering:

$$\mathrm{idf}_t=\ln\left(\frac{1+N}{1+\mathrm{df}_t}\right)+1.$$

After multiplying by raw counts, normalize each nonzero vector to Euclidean length 1. Even setting `smooth_idf=False` in scikit-learn retains the outer `+1`; that option alone would not reproduce our classroom formula.


In [19]:
software_idf, software_weights = tfidf(toy_counts, convention="smoothed_l2")
show_table(["term", "classroom IDF", "smoothed IDF"],
           [[t, f"{toy_idf[t]:.3f}", f"{software_idf[t]:.3f}"] for t in vocabulary])
print()
show_table(["document"] + vocabulary,
           [[f"D{i + 1}"] + [f"{row.get(t, 0):.3f}" for t in vocabulary]
            for i, row in enumerate(software_weights)])


term    classroom IDF  smoothed IDF
------  -------------  ------------
policy  0.000          1.000       
jobs    1.099          1.693       
wages   0.405          1.288       
prices  1.099          1.693       

document  policy  jobs   wages  prices
--------  ------  -----  -----  ------
D1        0.283   0.959  0.000  0.000 
D2        0.613   0.000  0.790  0.000 
D3        0.425   0.000  0.548  0.720 


## 6. Sensitivity on real speeches

Denny and Spirling (2018) show why preprocessing deserves attention in unsupervised analysis. Here we compare a small set of choices and inspect changing similarities. This exercise **does not implement their `preText` statistic** and does not identify a universally correct pipeline.

Keep the same four speeches from class 1 throughout. They are a selected teaching sample, not a basis for estimating party effects or historical trends. The original archive is not identified in the source material; see `data/README.md` for provenance.


In [20]:
# Find the repository data from the notebook directory, repository root,
# or another directory inside this checkout.
candidates = [folder / "data" / "sotu_sample.csv"
              for folder in [Path.cwd(), *Path.cwd().parents]]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Run from this repository; expected data/sotu_sample.csv")
with data_path.open(encoding="utf-8", newline="") as handle:
    speeches = list(csv.DictReader(handle))
labels = [f"{row['year']} {row['president']}" for row in speeches]
show_table(["year", "president", "party"],
           [[row['year'], row['president'], row['party']] for row in speeches])


year  president       party     
----  --------------  ----------
2008  George W. Bush  Republican
2010  Barack Obama    Democratic
2016  Barack Obama    Democratic
2020  Donald Trump    Republican


### Compare four pipelines

A is the baseline. B changes stopword removal; C adds bigrams to B; D weights B with the classroom TF-IDF formula. Compare **A versus B**, **B versus C**, and **B versus D** to separate these particular choices.

The custom stopword list is printed below and keeps `no`, `not`, and `never`. It is a small teaching list. Bigrams are made within approximate sentence boundaries after stopword removal. Like the earlier `not in work` example, these are adjacent *retained* words and need not be adjacent in the original sentence. The sentence splitter can make mistakes at abbreviations.


In [21]:
SPEECH_STOP = {"the", "a", "an", "and", "or", "of", "to", "in", "on", "for",
               "is", "are", "was", "were", "be", "been", "this", "that", "it",
               "as", "with", "by", "at", "from", "we", "our", "you", "your"}
print("Stopword list:", ", ".join(sorted(SPEECH_STOP)))

def speech_features(text, stopwords=None, add_bigrams=False):
    stopwords = set() if stopwords is None else stopwords
    result = Counter()
    for sentence in re.split(r"(?<=[.!?])\s+", text.strip()):
        tokens = remove_stopwords(word_tokens(sentence), stopwords)
        result.update(tokens)
        if add_bigrams:
            result.update(ngrams(tokens, 2))
    return result

A = [speech_features(row["text"]) for row in speeches]
B = [speech_features(row["text"], SPEECH_STOP) for row in speeches]
C = [speech_features(row["text"], SPEECH_STOP, add_bigrams=True) for row in speeches]
_, D = tfidf(B)
pipelines = {"A: all unigrams": A, "B: remove stopwords": B,
             "C: B plus bigrams": C, "D: TF-IDF of B": D}
summary = []
for name, vectors in pipelines.items():
    active = {term for row in vectors for term, value in row.items() if value > 0}
    empty = sum(not any(value > 0 for value in row.values()) for row in vectors)
    summary.append([name, len(active), empty])
show_table(["pipeline", "features with positive values", "empty rows"], summary)


Stopword list: a, an, and, are, as, at, be, been, by, for, from, in, is, it, of, on, or, our, that, the, this, to, was, we, were, with, you, your
pipeline             features with positive values  empty rows
-------------------  -----------------------------  ----------
A: all unigrams      3860                           0         
B: remove stopwords  3832                           0         
C: B plus bigrams    17000                          0         
D: TF-IDF of B       3458                           0         


The last table counts features with **positive values**, not merely allocated columns. Under classroom TF-IDF, terms present in all four speeches have weight zero. Adding bigrams also means a row sum is no longer a word-token count.


In [22]:
speech_pairs = list(combinations(range(len(speeches)), 2))
show_table(["pair"] + list(pipelines),
           [[f"{speeches[i]['year']}-{speeches[j]['year']}"]
            + [number(cosine(vectors[i], vectors[j])) for vectors in pipelines.values()]
            for i, j in speech_pairs])
print()
for name, vectors in pipelines.items():
    scored = [(cosine(vectors[i], vectors[j]), i, j) for i, j in speech_pairs]
    valid = [entry for entry in scored if entry[0] is not None]
    if valid:
        score, i, j = max(valid)
        print(f"{name}: closest pair = {speeches[i]['year']}-{speeches[j]['year']} ({score:.3f})")
    else:
        print(f"{name}: no pair has a defined cosine")


pair       A: all unigrams  B: remove stopwords  C: B plus bigrams  D: TF-IDF of B
---------  ---------------  -------------------  -----------------  --------------
2008-2010  0.949            0.731                0.661              0.087         
2008-2016  0.942            0.684                0.615              0.058         
2008-2020  0.952            0.660                0.589              0.083         
2010-2016  0.971            0.836                0.768              0.170         
2010-2020  0.934            0.698                0.631              0.070         
2016-2020  0.931            0.661                0.591              0.063         

A: all unigrams: closest pair = 2010-2016 (0.971)
B: remove stopwords: closest pair = 2010-2016 (0.836)
C: B plus bigrams: closest pair = 2010-2016 (0.768)
D: TF-IDF of B: closest pair = 2010-2016 (0.170)


**Check the supplied result.** All four pipelines select 2010–2016 as the closest pair, despite different cosine values. Some other pair orderings change. Check both rankings and the text behind them; this pattern does not establish a party effect.

### Read the features behind the comparison

Choose a speech and inspect the highest-valued features for each pipeline. A high TF-IDF feature is distinctive relative to **these four texts**; it need not be the speech's most important issue.


In [23]:
YEAR_TO_INSPECT = "2010"
selected_index = next(i for i, row in enumerate(speeches) if row["year"] == YEAR_TO_INSPECT)
for name, vectors in pipelines.items():
    ranked = sorted(((term, value) for term, value in vectors[selected_index].items() if value > 0),
                    key=lambda item: (-item[1], item[0]))[:8]
    print("\n" + name)
    show_table(["feature", "value"], [[term, f"{value:.3f}"] for term, value in ranked])

print("\nHighest-count bigrams in pipeline C:")
phrases = sorted(((term, value) for term, value in C[selected_index].items() if " " in term),
                 key=lambda item: (-item[1], item[0]))[:8]
show_table(["retained-token bigram", "count"], phrases)



A: all unigrams
feature  value  
-------  -------
the      337.000
to       239.000
and      235.000
of       167.000
that     147.000
we       140.000
a        125.000
our      120.000

B: remove stopwords
feature  value 
-------  ------
i        77.000
will     60.000
have     53.000
more     43.000
but      42.000
they     42.000
all      37.000
now      36.000

C: B plus bigrams
feature  value 
-------  ------
i        77.000
will     60.000
have     53.000
more     43.000
but      42.000
they     42.000
all      37.000
now      36.000

D: TF-IDF of B
feature         value 
--------------  ------
deficit         13.863
lobbyists       8.318 
values          8.318 
can't           7.625 
financial       7.625 
that's          7.480 
debt            6.931 
small-business  6.931 

Highest-count bigrams in pipeline C:
retained-token bigram  count
---------------------  -----
that's why             17   
american people        11   
i know                 11   
clean energy           1

In [24]:
TERM_TO_INSPECT = "jobs"  # Choose a single word from the feature lists.
selected_speech = speeches[selected_index]
sentences = re.split(r"(?<=[.!?])\s+", selected_speech["text"].strip())
matching = [sentence for sentence in sentences if TERM_TO_INSPECT.lower() in word_tokens(sentence)]
print(f"{labels[selected_index]}: {len(matching)} matched passages; showing up to 5\n")
for sentence in matching[:5]:
    print(" ".join(sentence.split()) + "\n")


2010 Barack Obama: 21 matched passages; showing up to 5

Now, as we stabilized the financial system, we also took steps to get our economy growing again, save as many jobs as possible, and help Americans who had become unemployed.

And we're on track to add another 1 1/2 million jobs to this total by the end of the year.

The plan that has made all of this possible, from the tax cuts to the jobs, is the Recovery Act.

Economists on the left and the right say this bill has helped save jobs and avert disaster.

That is why jobs must be our number-one focus in 2010, and that's why I'm calling for a new jobs bill tonight.



### Keep the denominator of a measure explicit

Compare the same exact `job`/`jobs` numerator with two denominators: all word tokens versus retained tokens after stopword removal. If the numerator stays unchanged, a higher rate under the second definition reflects the smaller denominator.


In [25]:
TARGET_WORDS = {"job", "jobs"}
rate_rows = []
for row in speeches:
    raw = word_tokens(row["text"])
    retained = remove_stopwords(raw, SPEECH_STOP)
    raw_count = sum(token in TARGET_WORDS for token in raw)
    kept_count = sum(token in TARGET_WORDS for token in retained)
    rate_rows.append([row["year"], raw_count, kept_count,
                      f"{1000 * raw_count / len(raw):.2f}" if raw else "undefined",
                      f"{1000 * kept_count / len(retained):.2f}" if retained else "undefined"])
show_table(["year", "raw mentions", "retained mentions", "per 1,000 raw", "per 1,000 retained"], rate_rows)


year  raw mentions  retained mentions  per 1,000 raw  per 1,000 retained
----  ------------  -----------------  -------------  ------------------
2008  6             6                  1.06           1.61              
2010  29            29                 4.03           5.88              
2016  19            19                 3.15           4.65              
2020  14            14                 2.25           3.26              


### Your measurement memo

Choose either **employment attention**, **stance toward an employment policy**, or **group identity**.

1. State your question and the unit of analysis.
2. Choose a pipeline and explain which distinctions it preserves.
3. Report one observed difference between two pipelines, using the tables above.
4. Read a relevant passage. Does the representation capture what you need?
5. State one limitation and one further check. Do not infer a party effect from these four speeches.

Your answer:

A stable result is not proof of validity. A changing result is a reason to investigate, not permission to choose whichever output best matches your expectations. The pipeline for out-of-sample prediction would also need to learn vocabulary and IDF on training documents only.


## References

- Denny, M. J., & Spirling, A. (2018). *Text Preprocessing for Unsupervised Learning: Why It Matters, When It Misleads, and What to Do About It*. Political Analysis, 26(2), 168–189. [Author manuscript](https://arthurspirling.org/documents/preprocessing.pdf).
- Manning, C. D., Raghavan, P., & Schütze, H. (2008). *Introduction to Information Retrieval*, Chapters 2 and 6. [Online book](https://nlp.stanford.edu/IR-book/).
- [scikit-learn TF-IDF conventions](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html).
- Examples adapted from CEU 2025 and MethodsNET 2026; local speech provenance is described in `data/README.md`.
